3. InsightFaceによる予測

In [29]:
import cv2
import numpy as np
import os

def process_video_opencv_dnn(video_path, target_face_path, output_path="output_labeled.mp4"):
    # --- 1. モデルのパス設定 (相対パス) ---
    model_det = "face_detection_yunet_2023mar.onnx"
    model_rec = "face_recognition_sface_2021dec.onnx"

    # ファイル存在確認
    for m in [model_det, model_rec]:
        if not os.path.exists(m):
            print(f"Error: モデルファイル {m} が見つかりません。")
            return

    # モデルのロード
    detector = cv2.FaceDetectorYN.create(model_det, "", (320, 320))
    recognizer = cv2.FaceRecognizerSF.create(model_rec, "")
    
    # 検出感度の調整 (0.9 -> 0.5に下げて検出しやすくする)
    detector.setScoreThreshold(0.5)

    # --- 2. 登録用顔画像の処理 (Registration) ---
    img_target = cv2.imread(target_face_path)
    if img_target is None:
        print(f"Error: 登録用画像が見つかりません: {target_face_path}")
        return

    h_t, w_t, _ = img_target.shape
    detector.setInputSize((w_t, h_t))
    result, face_target = detector.detect(img_target)

    # 顔検出失敗時の例外処理
    if face_target is None or len(face_target) == 0:
        print(f"Error: 登録画像 {target_face_path} から顔を検出できませんでした。別の画像で試してください。")
        return

    # 特徴量抽出
    face_align_target = recognizer.alignCrop(img_target, face_target[0])
    target_feature = recognizer.feature(face_align_target)
    print("Success: 登録顔画像の特徴量抽出が完了しました。")

    # --- 3. 動画処理の設定 ---
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: 動画ファイルが開けません: {video_path}")
        return

    # 動画の定量的パラメータ取得
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # コーデック
    
    # 保存用のVideoWriter設定
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # 検出器に動画サイズをセット
    detector.setInputSize((width, height))

    print(f"Analysis Started: {width}x{height} @ {fps}fps")
    print("Escキーで中断できます。")

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        # 動画フレーム内の顔検出
        _, faces = detector.detect(frame)

        if faces is not None:
            for face in faces:
                # 顔の位置合わせと特徴量抽出
                aligned_face = recognizer.alignCrop(frame, face)
                current_feature = recognizer.feature(aligned_face)

                # コサイン類似度の計算
                score = recognizer.match(target_feature, current_feature, cv2.FaceRecognizerSF_FR_COSINE)

                # 閾値判定 (0.36以上で同一人物とみなすのがSFaceの目安)
                # 環境に合わせて 0.30 ~ 0.40 で調整してください
                is_target = score > 0.30
                label = "Target" if is_target else "Unknown"
                color = (0, 255, 0) if is_target else (0, 0, 255) # 緑:本人, 赤:他人
                
                # 描画処理
                box = list(map(int, face[:4]))
                cv2.rectangle(frame, (box[0], box[1]), (box[0]+box[2], box[1]+box[3]), color, 2)
                cv2.putText(frame, f"{label}: {score:.2f}", (box[0], box[1]-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # 保存用書き出し
        out.write(frame)
        
        # プレビュー表示
        cv2.imshow('Face Labeling System', frame)
        if cv2.waitKey(1) & 0xFF == 27: break # Escキー

    # 後片付け
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Finished: 結果を {output_path} に保存しました。")

# --- 実行コード ---
video_path = "./movie/nogi_movie_001.mp4"
target_face_path = "./data_bing/000001.jpg"

process_video_opencv_dnn(video_path, target_face_path)

Success: 登録顔画像の特徴量抽出が完了しました。
Analysis Started: 360x640 @ 30fps
Escキーで中断できます。
Finished: 結果を output_labeled.mp4 に保存しました。


In [ ]:
import cv2
import numpy as np
import os

def process_video_multi_person(video_path, db_base_path, output_path="output_final.mp4"):
    # 1. モデルのロード
    detector = cv2.FaceDetectorYN.create("face_detection_yunet_2023mar.onnx", "", (320, 320))
    recognizer = cv2.FaceRecognizerSF.create("face_recognition_sface_2021dec.onnx", "")
    detector.setScoreThreshold(0.5)

    # 2. 特徴量データベース（Face DB）の構築
    # フォルダ名をキー、平均特徴量を値として保持
    face_db = {}
    target_folders = ["endou", "kaki"] # 画像フォルダ名

    print("--- データベース構築開始 ---")
    for label in target_folders:
        # 対象の画像フォルダとラベルを紐づけてパスを作成
        dir_path = os.path.join(db_base_path, label)
        if not os.path.exists(dir_path):
            print(f"Warning: {dir_path} が見つかりません。スキップします。")
            continue
        
        embeddings = []
        #指定したディレクトリ内にある「すべてのファイル名とフォルダ名の文字列」を Python の list 形式で返します。
        for img_name in os.listdir(dir_path):
            # 画像ファイルのみ処理
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            
            # cv2.imread()は画像ファイルを読み込み、数値の多次元配列（NumPy配列）に変換する関数。高さ×幅×チャンネル数（RGBの3次元配列）を読み込む
            img = cv2.imread(os.path.join(dir_path, img_name))
            if img is None: continue
            
            #detector.create 時に設定したデフォルト値 (320, 320) は、この行によって現在の画像の実際の解像度に上書きされる。
            detector.setInputSize((img.shape[1], img.shape[0]))
            # YuNet 検出器を実行し、結果を受け取っています。Python の慣習で _ は「戻り値はあるが、このコードでは使わない変数」を指します。
            # _は内部的なステータスコード（成功/失敗）などが入る
            # facesには検出された顔の情報の行列が格納。形状は[N, 15] の行列（N は検出された人数）。
            # 各行には行の中身: [x, y, w, h, 右目x, 右目y, 左目x, 左目y, ... 信頼度スコア] の計 15 個の数値が入る
            _, faces = detector.detect(img)
            
            # 検出された顔画像のEmbeddingを抽出してデータベースに登録
            if faces is not None:
                # 顔の正規化:検出された顔（faces[0]）は、首を傾けていたり、斜めを向いていたりします。
                # この関数は、顔を「正面」に向くよう回転・リサイズし、特定のサイズ（SFaceモデルでは通常 112ピクセル）に切り出します。
                aligned_face = recognizer.alignCrop(img, faces[0])
                # 特徴抽出:正規化された顔画像を、学習済みのディープニューラルネットワーク（CNN）に通し、その人の顔の特徴を凝縮した「数値の列（ベクトル）」に変換
                feat = recognizer.feature(aligned_face)
                # 抽出された128次元のベクトルをリストに追加します。
                embeddings.append(feat)
        
        if embeddings:
            # 50枚のベクトルの平均をとることでロバスト性を向上 ($E[X]$)
            face_db[label] = np.mean(embeddings, axis=0)
            print(f"Registered: {label} (Images: {len(embeddings)})")

    # 3. 動画処理の設定
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: 動画が開けません。")
        return

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    detector.setInputSize((width, height))

    print(f"--- 動画解析開始: {output_path} へ保存中 ---")

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        _, faces = detector.detect(frame)
        if faces is not None:
            for face in faces:
                aligned_face = recognizer.alignCrop(frame, face)
                current_feat = recognizer.feature(aligned_face)

                best_score = -1
                best_label = "Unknown"

                # DB内の全員とコサイン類似度で比較
                for label, target_feat in face_db.items():
                    score = recognizer.match(target_feat, current_feat, cv2.FaceRecognizerSF_FR_COSINE)
                    if score > best_score:
                        best_score = score
                        best_label = label

                # 閾値判定 (0.36がSFaceの推奨基準値)
                is_match = best_score > 0.36
                display_label = best_label if is_match else "Unknown"
                color = (0, 255, 0) if is_match else (0, 0, 255) # 緑:本人, 赤:Unknown
                
                # 描画
                box = list(map(int, face[:4]))
                cv2.rectangle(frame, (box[0], box[1]), (box[0]+box[2], box[1]+box[3]), color, 2)
                cv2.putText(frame, f"{display_label}: {best_score:.2f}", (box[0], box[1]-10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        out.write(frame)
        cv2.imshow('Recognition Test', frame)
        if cv2.waitKey(1) & 0xFF == 27: break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("解析が完了しました。")

# 実行コード
# 画像のディレクトリ構造に合わせて '.' (現在のフォルダ) を指定
process_video_multi_person("./movie/nogi_movie_001.mp4", "./")

--- データベース構築開始 ---
Registered: endou (Images: 15)
Registered: kaki (Images: 51)
--- 動画解析開始: output_final.mp4 へ保存中 ---
解析が完了しました。


4. 動画で1フレームごとに取り出す<br>
参考：https://qiita.com/tsuruQQ/items/c56efda298ee93e4f389<br>
参考：https://github.com/serengil/deepface

In [1]:
video_path = "./movie/nogi_movie_001.mp4"
output_dir = "./movie_frame"

# ディレクトリが存在しなければ作成
os.makedirs(output_dir, exist_ok=True)

print(f"処理対象: {video_path}")
print(f"ファイル存在確認: {os.path.exists(video_path)}")

video = cv2.VideoCapture(video_path)

# ビデオが正しく開かれたか確認
if not video.isOpened():
    print("エラー：動画ファイルが開けません")
else:
    fps = int(video.get(cv2.CAP_PROP_FPS))
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"FPS: {fps}, 総フレーム数: {total_frames}, 解像度: {width}×{height}")
    
    frame_count = 0
    saved_frames = []

    while True:
        success, frame = video.read() 
        if not success:
            break  # 動画の終了

        # 1秒ごとにフレームを保存
        if frame_count % fps == 0:
            frame_time_sec = int(frame_count / fps)
            frame_file = os.path.join(output_dir, f"frame_{frame_time_sec:05d}.jpg")
            # リサイズなしで元のサイズのまま保存
            cv2.imwrite(frame_file, frame)
            saved_frames.append((frame_time_sec, frame_file))
        
        frame_count += 1

    video.release()
    print(f"✅ 処理完了！保存されたフレーム数: {len(saved_frames)}")

NameError: name 'os' is not defined

5. フレームごとに

In [3]:
import cv2
import numpy as np
import os
from deepface import DeepFace

def get_deepface_db(base_path, model_name='Facenet'):
    face_db = {}
    target_folders = ["endou", "kaki"]
    
    print("--- DeepFace 特徴量抽出開始 ---")
    for label in target_folders:
        dir_path = os.path.join(base_path, label)
        if not os.path.exists(dir_path): continue
        
        embeddings = []
        for img_name in os.listdir(dir_path):
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            try:
                # 登録時は精度の高い検出器を使用
                objs = DeepFace.represent(img_path=os.path.join(dir_path, img_name), 
                                          model_name=model_name, 
                                          detector_backend='retinaface', 
                                          enforce_detection=False)
                for obj in objs:
                    embeddings.append(obj["embedding"])
            except:
                continue
        
        if embeddings:
            face_db[label] = np.mean(embeddings, axis=0)
            print(f"Registered: {label} ({len(embeddings)} images)")
            
    return face_db

def process_video_deepface_multi(video_path, db_base_path, output_path="output_deepface_fast.mp4"):
    # --- パラメータ調整 ---
    MODEL_NAME = 'Facenet'
    DETECTOR = 'retinaface' # opencvからretinafaceに変更 (重要)
    THRESHOLD = 0.40        # 識別しきい値
    SKIP_FRAMES = 10        # 10倍速
    CONFIDENCE_MIN = 0.3    # 検出信頼度の下限 (緩和)
    
    face_db = get_deepface_db(db_base_path, model_name=MODEL_NAME)
    
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    print(f"--- 動画解析開始 (Detector: {DETECTOR}) ---")
    
    frame_idx = 0
    last_results = [] 

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        # AI推論（SKIP_FRAMESごと）
        if frame_idx % SKIP_FRAMES == 0:
            try:
                # 検出器を強化版に変更
                last_results = DeepFace.represent(img_path=frame, 
                                             model_name=MODEL_NAME, 
                                             detector_backend=DETECTOR, 
                                             enforce_detection=False)
            except Exception as e:
                last_results = []

        # 描画処理
        if last_results:
            for res in last_results:
                # 検出信頼度のチェック（少しでも顔っぽければ通す）
                conf = res.get("face_confidence", 0)
                if conf > CONFIDENCE_MIN:
                    current_feat = np.array(res["embedding"])
                    
                    # 識別計算
                    best_score = -1
                    best_label = "Unknown"
                    for label, target_feat in face_db.items():
                        # コサイン類似度
                        score = np.dot(target_feat, current_feat) / (np.linalg.norm(target_feat) * np.linalg.norm(current_feat))
                        if score > best_score:
                            best_score = score
                            best_label = label

                    # ラベリング
                    is_match = best_score > (1 - THRESHOLD)
                    display_label = best_label if is_match else "Unknown"
                    color = (0, 255, 0) if is_match else (0, 0, 255)

                    # 行列への上書き（枠とテキスト）
                    box = res["facial_area"]
                    cv2.rectangle(frame, (box['x'], box['y']), (box['x']+box['w'], box['y']+box['h']), color, 2)
                    cv2.putText(frame, f"{display_label}: {best_score:.2f}", (box['x'], box['y']-10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                    
                    # デバッグ用ログ出力
                    if frame_idx % 100 == 0:
                        print(f"Frame {frame_idx}: Found {display_label} (Conf: {conf:.2f}, Score: {best_score:.2f})")

        out.write(frame)
        frame_idx += 1
        
        if frame_idx % 100 == 0:
            print(f"Progress: {frame_idx} frames processed...")

    cap.release()
    out.release()
    print(f"Completed. Results saved to {output_path}")

# 実行
process_video_deepface_multi("./movie/nogi_movie_001.mp4", "./")

--- DeepFace 特徴量抽出開始 ---
Registered: endou (15 images)
Registered: kaki (59 images)
--- 動画解析開始 (Detector: retinaface) ---
Frame 0: Found endou (Conf: 1.00, Score: 0.75)
Progress: 100 frames processed...
Frame 100: Found kaki (Conf: 1.00, Score: 0.67)
Progress: 200 frames processed...
Frame 200: Found endou (Conf: 1.00, Score: 0.87)
Progress: 300 frames processed...
Frame 300: Found endou (Conf: 1.00, Score: 0.86)
Progress: 400 frames processed...
Frame 400: Found kaki (Conf: 1.00, Score: 0.72)
Frame 400: Found kaki (Conf: 1.00, Score: 0.72)
Progress: 500 frames processed...
Frame 500: Found kaki (Conf: 1.00, Score: 0.75)
Progress: 600 frames processed...
Frame 600: Found endou (Conf: 1.00, Score: 0.79)
Progress: 700 frames processed...
Frame 700: Found kaki (Conf: 1.00, Score: 0.77)
Progress: 800 frames processed...
Frame 800: Found kaki (Conf: 1.00, Score: 0.64)
Frame 800: Found kaki (Conf: 1.00, Score: 0.65)
Progress: 900 frames processed...
Frame 900: Found kaki (Conf: 1.00, Score: 

## 評価したい<br>
・まずは動画をフレームごとに分けて、画像をフレームごとに保存する。またそれに対してフレームIDを付与する<br>
・フレームIDごとにどのラベルなのかを考えて、付与していく。<br>
└この場合、顔検出精度（人数を正しく検出したか）とラベル精度（各顔の順序含め完全一致したか）の二つを評価する<br>

### 動画を１フレームごとに切り出す

In [15]:
def extract_frames(video_path, output_dir):
    # ディレクトリが存在しなければ作成
    os.makedirs(output_dir, exist_ok=True)

    print(f"処理対象: {video_path}")
    print(f"ファイル存在確認: {os.path.exists(video_path)}")

    video = cv2.VideoCapture(video_path)

    # ビデオが正しく開かれたか確認
    if not video.isOpened():
        print("エラー：動画ファイルが開けません")
    else:
        fps = int(video.get(cv2.CAP_PROP_FPS))
        total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f"FPS: {fps}, 総フレーム数: {total_frames}, 解像度: {width}×{height}")
        
        frame_count = 0
        saved_frames = []

        while True:
            success, frame = video.read() 
            if not success:
                break  # 動画の終了

            # 1秒ごとにフレームを保存
            if frame_count % fps == 0:
                frame_time_sec = int(frame_count / fps)
                frame_file = os.path.join(output_dir, f"frame_{frame_time_sec:05d}.jpg")
                # リサイズなしで元のサイズのまま保存
                cv2.imwrite(frame_file, frame)
                saved_frames.append((frame_time_sec, frame_file))
            
            frame_count += 1

        video.release()
        print(f"✅ 処理完了！保存されたフレーム数: {len(saved_frames)}")

### 元画像データの特徴量をface_dbに格納

In [ ]:
import cv2
import numpy as np
import os

def create_embeddings_db(db_base_path):
    # 1. モデルのロード
    detector = cv2.FaceDetectorYN.create("face_detection_yunet_2023mar.onnx", "", (320, 320))
    recognizer = cv2.FaceRecognizerSF.create("face_recognition_sface_2021dec.onnx", "")
    detector.setScoreThreshold(0.5)

    # 2. 特徴量データベース（Face DB）の構築
    # フォルダ名をキー、平均特徴量を値として保持
    face_db = {}
    target_folders = ["endou", "kaki"] # 画像フォルダ名

    print("--- データベース構築開始 ---")
    for label in target_folders:
        # 対象の画像フォルダとラベルを紐づけてパスを作成
        dir_path = os.path.join(db_base_path, label)
        if not os.path.exists(dir_path):
            print(f"Warning: {dir_path} が見つかりません。スキップします。")
            continue
        
        embeddings = []
        #指定したディレクトリ内にある「すべてのファイル名とフォルダ名の文字列」を Python の list 形式で返します。
        for img_name in os.listdir(dir_path):
            # 画像ファイルのみ処理
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            
            # cv2.imread()は画像ファイルを読み込み、数値の多次元配列（NumPy配列）に変換する関数。高さ×幅×チャンネル数（RGBの3次元配列）を読み込む
            img = cv2.imread(os.path.join(dir_path, img_name))
            if img is None: continue
            
            #detector.create 時に設定したデフォルト値 (320, 320) は、この行によって現在の画像の実際の解像度に上書きされる。
            detector.setInputSize((img.shape[1], img.shape[0]))
            # YuNet 検出器を実行し、結果を受け取っています。Python の慣習で _ は「戻り値はあるが、このコードでは使わない変数」を指します。
            # _は内部的なステータスコード（成功/失敗）などが入る
            # facesには検出された顔の情報の行列が格納。形状は[N, 15] の行列（N は検出された人数）。
            # 各行には行の中身: [x, y, w, h, 右目x, 右目y, 左目x, 左目y, ... 信頼度スコア] の計 15 個の数値が入る
            _, faces = detector.detect(img)
            
            # 検出された顔画像のEmbeddingを抽出してデータベースに登録
            if faces is not None:
                # 顔の正規化:検出された顔（faces[0]）は、首を傾けていたり、斜めを向いていたりします。
                # この関数は、顔を「正面」に向くよう回転・リサイズし、特定のサイズ（SFaceモデルでは通常 112ピクセル）に切り出します。
                aligned_face = recognizer.alignCrop(img, faces[0])
                # 特徴抽出:正規化された顔画像を、学習済みのディープニューラルネットワーク（CNN）に通し、その人の顔の特徴を凝縮した「数値の列（ベクトル）」に変換
                feat = recognizer.feature(aligned_face)
                # 抽出された128次元のベクトルをリストに追加します。
                embeddings.append(feat)
        
        if embeddings:
            # 50枚のベクトルの平均をとることでロバスト性を向上 ($E[X]$)
            face_db[label] = np.mean(embeddings, axis=0)
            print(f"Registered: {label} (Images: {len(embeddings)})")

    return face_db

### フレームごとに出た画像を一つ一つ取り出し、評価をしてdataframeに格納する

In [ ]:
frame_input_dir = "./movie_frame/"

os.makedirs(output_dir, exist_ok=True)

print(f"処理対象: {video_path}")
print(f"ファイル存在確認: {os.path.exists(video_path)}")

In [ ]:
video_path = "./movie/nogi_movie_001.mp4"
output_dir = "./movie_frame"
db_base_path="./"

extract_frames(video_path, output_dir)
face_db = create_embeddings_db(db_base_path=db_base_path)


--- データベース構築開始 ---
Registered: endou (Images: 15)
Registered: kaki (Images: 51)


In [ ]:


db_base_path = "./"
output_path = "./output.mp4"

# --- パラメータ調整 ---
MODEL_NAME = 'Facenet'
DETECTOR = 'retinaface' # opencvからretinafaceに変更 (重要)
THRESHOLD = 0.40        # 識別しきい値
SKIP_FRAMES = 10        # 10倍速
CONFIDENCE_MIN = 0.3    # 検出信頼度の下限 (緩和)

face_db = get_deepface_db(db_base_path, model_name=MODEL_NAME)
print(face_db)

--- DeepFace 特徴量抽出開始 ---
./endou
000001.jpg
000002.jpg
000003.jpg
000004.jpg
000005.jpg
000006.jpg
000007.jpg
000008.jpg
000009.jpg
000010.jpg
000011.jpg
000012.jpg
000013.jpg
000014.jpg
000015.jpg
./kaki
000001.jpg
000002.jpg
000003.jpg
000004.jpg
000005.jpg
000006.jpg
000007.jpg
000008.jpg
000009.jpg
000010.jpg
000011.jpg
000012.jpg
000013.jpg
000014.jpg
000015.jpg
000016.jpg
000017.jpg
000018.jpg
000019.jpg
000020.jpg
000021.jpg
000022.jpg
000023.jpg
000024.jpg
000025.jpg
000026.jpg
000027.jpg
000028.jpg
000029.jpg
000030.jpg
000031.jpg
000032.jpg
000033.jpg
000034.jpg
000035.jpg
000036.jpg
000037.jpg
000038.jpg
000039.jpg
000040.jpg
000041.jpg
000042.jpg
000043.jpg
000044.jpg
000045.jpg
000046.jpg
000047.jpg
000048.jpg
000049.jpg
000050.jpg
000051.jpg
{}


## 6. 精度評価フレームワーク

process_video_multi_person 関数の精度を評価します


In [ ]:
import cv2
import numpy as np
from collections import defaultdict
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def create_test_ground_truth(video_path, output_file="ground_truth.csv"):
    """
    動画をフレームごとに表示し、各フレームの正解ラベルを手動でアノテーション
    形式: frame_number, person_name, face_count (複数人の場合は改行して記録)
    """
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_count = 0
    annotations = []
    
    print("フレームごとに正解ラベルを入力してください")
    print("形式: フレーム番号, 人物名 (複数人はカンマ区切り)、または 'skip' でスキップ")
    print("('quit' で終了)\n")
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        
        if frame_count % fps == 0:  # 1秒ごと
            cv2.imshow('Annotation Helper', frame)
            frame_sec = frame_count // fps
            
            user_input = input(f"Frame {frame_sec}sec ({frame_count}): ")
            
            if user_input.lower() == 'quit':
                break
            elif user_input.lower() == 'skip':
                pass
            else:
                # カンマ区切りで複数人対応
                labels = [l.strip() for l in user_input.split(',')]
                for label in labels:
                    if label:
                        annotations.append({'frame': frame_sec, 'person': label})
        
        frame_count += 1
    
    cap.release()
    cv2.destroyAllWindows()
    
    # CSVに保存
    df = pd.DataFrame(annotations)
    df.to_csv(output_file, index=False)
    print(f"\n✅ {len(annotations)} 件のアノテーションを {output_file} に保存しました")
    return df

def evaluate_video_recognition(video_path, db_base_path, ground_truth_file, 
                                model_name='Facenet', threshold=0.40, skip_frames=10):
    """
    認識結果と正解ラベルを比較して精度を評価
    """
    face_db = get_deepface_db(db_base_path, model_name=model_name)
    
    # 正解データの読み込み
    gt_df = pd.read_csv(ground_truth_file)
    gt_dict = defaultdict(list)
    for _, row in gt_df.iterrows():
        gt_dict[int(row['frame'])].append(row['person'])
    
    # 動画の処理
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    predictions = []  # (frame_sec, predicted_label)
    frame_idx = 0
    frame_sec = 0
    detected_faces = defaultdict(list)  # frame_sec -> [predicted_labels]
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        
        frame_sec = frame_idx / fps
        
        # 定期的に認識実行
        if frame_idx % (skip_frames * fps) == 0:  # skip_frames秒ごと
            try:
                results = DeepFace.represent(img_path=frame, 
                                            model_name=model_name, 
                                            detector_backend='retinaface', 
                                            enforce_detection=False)
                
                frame_sec_int = int(frame_sec)
                for res in results:
                    conf = res.get("face_confidence", 0)
                    if conf > 0.3:
                        current_feat = np.array(res["embedding"])
                        
                        best_score = -1
                        best_label = "Unknown"
                        for label, target_feat in face_db.items():
                            score = np.dot(target_feat, current_feat) / \
                                   (np.linalg.norm(target_feat) * np.linalg.norm(current_feat))
                            if score > best_score:
                                best_score = score
                                best_label = label
                        
                        is_match = best_score > (1 - threshold)
                        final_label = best_label if is_match else "Unknown"
                        detected_faces[frame_sec_int].append(final_label)
                        predictions.append((frame_sec_int, final_label))
            
            except:
                detected_faces[frame_sec_int].append("Error")
        
        frame_idx += 1
    
    cap.release()
    
    # 評価メトリクスの計算
    print("\n=== 精度評価結果 ===\n")
    
    all_true_labels = []
    all_pred_labels = []
    
    for frame_sec in sorted(gt_dict.keys()):
        if frame_sec in detected_faces:
            gt_labels = gt_dict[frame_sec]
            pred_labels = detected_faces[frame_sec]
            
            # 同じ数の予測と正解がある場合のみ比較（シンプル版）
            if len(pred_labels) == len(gt_labels):
                for gt, pred in zip(sorted(gt_labels), sorted(pred_labels)):
                    all_true_labels.append(gt)
                    all_pred_labels.append(pred)
                    print(f"Frame {frame_sec}sec: GT={gt}, PRED={pred}")
    
    if all_true_labels:
        print(f"\n📊 テスト対象フレーム数: {len(all_true_labels)}")
        print(f"正解率 (Accuracy): {accuracy_score(all_true_labels, all_pred_labels):.3f}")
        print("\n" + classification_report(all_true_labels, all_pred_labels))
        
        # 混同行列の可視化
        cm = confusion_matrix(all_true_labels, all_pred_labels)
        labels = sorted(set(all_true_labels + all_pred_labels))
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.title('Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️ 比較可能なデータがありません")
    
    return all_true_labels, all_pred_labels

# 使用例
# step 1: 正解ラベルの作成（初回のみ）
# create_test_ground_truth("./movie/nogi_movie_001.mp4", "ground_truth.csv")

# step 2: 精度評価
# evaluate_video_recognition("./movie/nogi_movie_001.mp4", "./", "ground_truth.csv")


### ステップ 1: 正解ラベルの準備

以下いずれかの方法で正解データを用意：

**方法A: インタラクティブにアノテーション**
```python
create_test_ground_truth("./movie/nogi_movie_001.mp4", "ground_truth.csv")
```

**方法B: CSVファイルを手動で作成**
- 形式: `frame,person` の 2 列
- 例：
```
frame,person
0,endou
0,kaki
5,endou
10,kaki
```


In [ ]:
# 例：ground_truth.csv を手動で作成した場合
# 実行してみる
true_labels, pred_labels = evaluate_video_recognition(
    video_path="./movie/nogi_movie_001.mp4",
    db_base_path="./",
    ground_truth_file="ground_truth.csv",
    model_name='Facenet',
    threshold=0.40,
    skip_frames=10
)

### ステップ 2: 複数人対応の精度評価

ground_truth.csv の形式：
```
frame,persons
0,endou|kaki
5,endou
10,kaki|unknown
```

パイプ記号 `|` で複数人を区切る


In [ ]:
def evaluate_video_recognition_multi_person(video_path, db_base_path, ground_truth_file, 
                                             model_name='Facenet', threshold=0.40, skip_frames=10):
    """
    複数人が映っているケースに対応した精度評価
    
    評価方式: 
    - フレーム内で検出される全顔の予測ラベル「セット」と正解ラベル「セット」を比較
    - 例: 正解={endou, kaki}, 予測={endou, kaki} → 正解
    - 例: 正解={endou}, 予測={endou, kaki} → 不正解（余計な検出）
    """
    face_db = get_deepface_db(db_base_path, model_name=model_name)
    
    # 正解データの読み込み
    gt_df = pd.read_csv(ground_truth_file)
    gt_dict = {}
    for _, row in gt_df.iterrows():
        frame = int(row['frame'])
        # パイプで複数人を分割し、セット化
        persons = set([p.strip() for p in str(row['persons']).split('|') if p.strip()])
        gt_dict[frame] = persons
    
    # 動画の処理
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    frame_results = []  # フレームごとの結果
    frame_idx = 0
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        
        frame_sec = int(frame_idx / fps)
        
        # 定期的に認識実行
        if frame_idx % (skip_frames * fps) == 0 and frame_sec in gt_dict:
            try:
                results = DeepFace.represent(img_path=frame, 
                                            model_name=model_name, 
                                            detector_backend='retinaface', 
                                            enforce_detection=False)
                
                predicted_persons = set()
                face_details = []
                
                for res in results:
                    conf = res.get("face_confidence", 0)
                    if conf > 0.3:
                        current_feat = np.array(res["embedding"])
                        
                        best_score = -1
                        best_label = "Unknown"
                        for label, target_feat in face_db.items():
                            score = np.dot(target_feat, current_feat) / \
                                   (np.linalg.norm(target_feat) * np.linalg.norm(current_feat))
                            if score > best_score:
                                best_score = score
                                best_label = label
                        
                        is_match = best_score > (1 - threshold)
                        final_label = best_label if is_match else "Unknown"
                        predicted_persons.add(final_label)
                        
                        face_details.append({
                            'label': final_label,
                            'score': best_score,
                            'confidence': conf
                        })
                
                gt_persons = gt_dict[frame_sec]
                is_correct = predicted_persons == gt_persons
                
                frame_results.append({
                    'frame': frame_sec,
                    'gt_persons': gt_persons,
                    'pred_persons': predicted_persons,
                    'correct': is_correct,
                    'face_count': len(face_details),
                    'details': face_details
                })
                
                print(f"Frame {frame_sec}s | GT: {sorted(gt_persons)} | PRED: {sorted(predicted_persons)} | {'✓' if is_correct else '✗'}")
            
            except Exception as e:
                frame_results.append({
                    'frame': frame_sec,
                    'gt_persons': gt_dict[frame_sec],
                    'pred_persons': set(),
                    'correct': False,
                    'face_count': 0,
                    'details': []
                })
        
        frame_idx += 1
    
    cap.release()
    
    # 分析
    print(f"\n=== 複数人対応 精度評価結果 ===")
    print(f"評価フレーム数: {len(frame_results)}")
    
    if frame_results:
        correct_frames = sum(1 for r in frame_results if r['correct'])
        accuracy = correct_frames / len(frame_results)
        print(f"フレーム正解率: {accuracy:.3f} ({correct_frames}/{len(frame_results)})")
        
        # 誤分類の詳細
        incorrect = [r for r in frame_results if not r['correct']]
        if incorrect:
            print(f"\n❌ {len(incorrect)} フレームで誤分類:")
            for r in incorrect:
                print(f"  Frame {r['frame']}: 正解={sorted(r['gt_persons'])} → 予測={sorted(r['pred_persons'])}")
    
    return pd.DataFrame(frame_results)

# 使用例（ground_truth.csv が複数人フォーマット対応の場合）
# results_df = evaluate_video_recognition_multi_person(
#     "./movie/nogi_movie_001.mp4", 
#     "./", 
#     "ground_truth.csv"
# )


### 方法2: テスト画像での複数人検出評価

複数人が映っているテスト画像セットを用意して精度を測定


In [ ]:
def evaluate_multi_person_images(test_images_dir="./test_multi_person", 
                                 db_base_path="./", 
                                 model_name='Facenet', 
                                 threshold=0.40):
    """
    複数人が映っているテスト画像の評価
    
    使用方法:
    1. test_multi_person/ フォルダ内に画像を配置
    2. CSVファイル (test_annotations.csv) を作成
       形式: image_name, persons
       例: pair_001.jpg, endou|kaki
    """
    face_db = get_deepface_db(db_base_path, model_name=model_name)
    
    # アノテーションCSVを読み込み
    anno_file = os.path.join(test_images_dir, "annotations.csv")
    if not os.path.exists(anno_file):
        print(f"⚠️ {anno_file} が見つかりません")
        return None
    
    anno_df = pd.read_csv(anno_file)
    
    results = []
    correct_count = 0
    
    print("=== 複数人テスト画像の精度評価 ===\n")
    
    for _, row in anno_df.iterrows():
        img_name = row['image_name']
        img_path = os.path.join(test_images_dir, img_name)
        
        if not os.path.exists(img_path):
            print(f"❌ {img_name} が見つかりません")
            continue
        
        # 正解ラベルをパースして
        gt_persons = set([p.strip() for p in str(row['persons']).split('|') if p.strip()])
        
        try:
            # 画像から複数顔を検出
            objs = DeepFace.represent(img_path=img_path, 
                                     model_name=model_name, 
                                     detector_backend='retinaface', 
                                     enforce_detection=False)
            
            pred_persons = set()
            face_details = []
            
            for obj in objs:
                conf = obj.get("face_confidence", 0)
                if conf > 0.3:
                    current_feat = np.array(obj["embedding"])
                    
                    best_score = -1
                    best_label = "Unknown"
                    for label, target_feat in face_db.items():
                        score = np.dot(target_feat, current_feat) / \
                               (np.linalg.norm(target_feat) * np.linalg.norm(current_feat))
                        if score > best_score:
                            best_score = score
                            best_label = label
                    
                    is_match = best_score > (1 - threshold)
                    final_label = best_label if is_match else "Unknown"
                    pred_persons.add(final_label)
                    
                    face_details.append({
                        'label': final_label,
                        'score': best_score,
                        'confidence': conf
                    })
            
            is_correct = pred_persons == gt_persons
            if is_correct:
                correct_count += 1
            
            results.append({
                'image': img_name,
                'gt_persons': sorted(gt_persons),
                'pred_persons': sorted(pred_persons),
                'correct': is_correct,
                'face_count': len(face_details)
            })
            
            status = "✓" if is_correct else "✗"
            print(f"{status} {img_name}: GT={sorted(gt_persons)} → PRED={sorted(pred_persons)}")
        
        except Exception as e:
            print(f"❌ {img_name}: エラー - {str(e)}")
            results.append({
                'image': img_name,
                'gt_persons': sorted(gt_persons),
                'pred_persons': [],
                'correct': False,
                'face_count': 0
            })
    
    # 結果集計
    print(f"\n📊 テスト画像数: {len(results)}")
    print(f"正解数: {correct_count}/{len(results)} ({100*correct_count/len(results):.1f}%)")
    
    results_df = pd.DataFrame(results)
    print("\n" + results_df.to_string(index=False))
    
    return results_df

# 使用前に以下を準備:
# 1. ./test_multi_person/ フォルダを作成
# 2. 複数人が映った jpg/png ファイルを配置
# 3. annotations.csv を作成:
#    image_name,persons
#    pair_001.jpg,endou|kaki
#    pair_002.jpg,endou
#    kaki_only.jpg,kaki

# results_df = evaluate_multi_person_images("./test_multi_person")


### 参考: 複数人ケースの3つの評価指標

| 指標 | 説明 | 例 |
|------|------|-----|
| **顔検出精度** | 正解の人数を正しく検出できたか | 正解2人 → 検出2人 = ○ |
| **セット照合精度** | フレーム内の全顔セットが完全一致するか | {A,B} == {A,B} = ○ |
| **個別ラベル精度** | 各顔の識別ラベルが正確か（人数違う場合も評価可能） | 正解[A,B] vs 予測[A,C] = 50% |


In [ ]:
def evaluate_multi_person_detailed(test_images_dir="./test_multi_person", 
                                   db_base_path="./", 
                                   model_name='Facenet', 
                                   threshold=0.40):
    """
    複数人テスト画像の詳細評価（3つの精度指標）
    
    1. 顔検出精度: 正解数と完全一致 → 人数が合っているか
    2. セット照合精度: セット一致度 → 正確な識別ができているか  
    3. 個別ラベル精度: ラベル一致度 → 各顔の識別スコア
    """
    face_db = get_deepface_db(db_base_path, model_name=model_name)
    
    anno_file = os.path.join(test_images_dir, "annotations.csv")
    if not os.path.exists(anno_file):
        print(f"⚠️ {anno_file} が見つかりません")
        return None
    
    anno_df = pd.read_csv(anno_file)
    
    detection_correct = 0  # 人数が正しい
    set_correct = 0        # セットが完全一致
    label_correct = 0      # 個別ラベルが一致
    
    detailed_results = []
    
    print("=== 複数人テスト画像 詳細評価 ===\n")
    print(f"{'Image':<20} {'正解':<15} {'予測':<15} {'検出':<5} {'セット':<5} {'ラベル':<6}")
    print("-" * 75)
    
    for _, row in anno_df.iterrows():
        img_name = row['image_name']
        img_path = os.path.join(test_images_dir, img_name)
        
        if not os.path.exists(img_path):
            continue
        
        gt_persons = sorted([p.strip() for p in str(row['persons']).split('|') if p.strip()])
        
        try:
            objs = DeepFace.represent(img_path=img_path, 
                                     model_name=model_name, 
                                     detector_backend='retinaface', 
                                     enforce_detection=False)
            
            pred_persons = []
            
            for obj in objs:
                conf = obj.get("face_confidence", 0)
                if conf > 0.3:
                    current_feat = np.array(obj["embedding"])
                    
                    best_score = -1
                    best_label = "Unknown"
                    for label, target_feat in face_db.items():
                        score = np.dot(target_feat, current_feat) / \
                               (np.linalg.norm(target_feat) * np.linalg.norm(current_feat))
                        if score > best_score:
                            best_score = score
                            best_label = label
                    
                    is_match = best_score > (1 - threshold)
                    final_label = best_label if is_match else "Unknown"
                    pred_persons.append(final_label)
            
            pred_persons = sorted(pred_persons)
            
            # 3つの評価指標
            detection_ok = (len(pred_persons) == len(gt_persons))  # 人数一致
            set_ok = (set(pred_persons) == set(gt_persons))        # セット一致
            label_ok = (pred_persons == gt_persons)                # 順序含め完全一致
            
            if detection_ok:
                detection_correct += 1
            if set_ok:
                set_correct += 1
            if label_ok:
                label_correct += 1
            
            # 表示
            det_mark = "✓" if detection_ok else "✗"
            set_mark = "✓" if set_ok else "✗"
            label_mark = "✓" if label_ok else "✗"
            
            gt_str = '|'.join(gt_persons)
            pred_str = '|'.join(pred_persons)
            
            print(f"{img_name:<20} {gt_str:<15} {pred_str:<15} {det_mark:<5} {set_mark:<5} {label_mark:<6}")
            
            detailed_results.append({
                'image': img_name,
                'gt': gt_str,
                'pred': pred_str,
                'detection': detection_ok,
                'set_match': set_ok,
                'label_match': label_ok
            })
        
        except:
            print(f"{img_name:<20} {'Error':<15}")
    
    # 集計
    total = len(anno_df)
    print("\n" + "="*50)
    print(f"📊 集計結果 (計 {total} 画像)")
    print("="*50)
    print(f"顔検出精度 (人数一致):    {detection_correct}/{total} ({100*detection_correct/total:.1f}%)")
    print(f"セット照合精度 (順不同):    {set_correct}/{total} ({100*set_correct/total:.1f}%)")
    print(f"ラベル精度 (順序含む完全一致): {label_correct}/{total} ({100*label_correct/total:.1f}%)")
    
    return pd.DataFrame(detailed_results)

# 使用例
# detailed_results = evaluate_multi_person_detailed("./test_multi_person")


### 📝 複数人ケースの評価フロー

**準備1: テスト画像セットの作成**
```
test_multi_person/
├── single_endou.jpg        （1人のみ）
├── single_kaki.jpg         （1人のみ）
├── pair_001.jpg            （2人一緒）
├── pair_002.jpg            （2人一緒）
└── annotations.csv
```

**準備2: annotations.csv の作成**
```csv
image_name,persons
single_endou.jpg,endou
single_kaki.jpg,kaki
pair_001.jpg,endou|kaki
pair_002.jpg,endou|kaki
```

**実行: 詳細評価で3つの指標を確認**
```python
detailed_results = evaluate_multi_person_detailed("./test_multi_person")
```

**出力例:**
```
顔検出精度 (人数一致):    4/4 (100.0%)
セット照合精度 (順不同):    4/4 (100.0%)
ラベル精度 (順序含む完全一致): 3/4 (75.0%)
```


### 別案: 画像ベースの簡易精度テスト

動画ではなく、既知の画像ファイルに対して直接精度測定（より簡単）


In [ ]:
def evaluate_image_recognition(db_base_path="./", model_name='Facenet', threshold=0.40):
    """
    各人物フォルダの画像に対して、フォルダ名をGTとして精度測定
    （既知の画像に対する認識精度を素早く確認できる）
    """
    face_db = get_deepface_db(db_base_path, model_name=model_name)
    target_folders = ["endou", "kaki"]
    
    all_true_labels = []
    all_pred_labels = []
    detailed_results = []
    
    print("=== 画像ベースの精度評価開始 ===\n")
    
    for true_label in target_folders:
        dir_path = os.path.join(db_base_path, true_label)
        if not os.path.exists(dir_path):
            continue
        
        correct_count = 0
        total_count = 0
        
        for img_name in os.listdir(dir_path):
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
            
            img_path = os.path.join(dir_path, img_name)
            total_count += 1
            
            try:
                objs = DeepFace.represent(img_path=img_path, 
                                         model_name=model_name, 
                                         detector_backend='retinaface', 
                                         enforce_detection=False)
                
                if objs:
                    current_feat = np.array(objs[0]["embedding"])
                    
                    best_score = -1
                    best_label = "Unknown"
                    for label, target_feat in face_db.items():
                        score = np.dot(target_feat, current_feat) / \
                               (np.linalg.norm(target_feat) * np.linalg.norm(current_feat))
                        if score > best_score:
                            best_score = score
                            best_label = label
                    
                    is_match = best_score > (1 - threshold)
                    pred_label = best_label if is_match else "Unknown"
                    
                    if pred_label == true_label:
                        correct_count += 1
                    
                    all_true_labels.append(true_label)
                    all_pred_labels.append(pred_label)
                    detailed_results.append({
                        'image': img_name,
                        'true_label': true_label,
                        'pred_label': pred_label,
                        'score': best_score,
                        'correct': pred_label == true_label
                    })
            except:
                all_true_labels.append(true_label)
                all_pred_labels.append("Error")
                detailed_results.append({
                    'image': img_name,
                    'true_label': true_label,
                    'pred_label': "Error",
                    'score': np.nan,
                    'correct': False
                })
        
        print(f"{true_label}: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    
    # 全体の評価メトリクス
    print(f"\n📊 全体の正解率: {accuracy_score(all_true_labels, all_pred_labels):.3f}")
    print("\n" + classification_report(all_true_labels, all_pred_labels, zero_division=0))
    
    # 混同行列
    if len(set(all_true_labels)) > 1:
        cm = confusion_matrix(all_true_labels, all_pred_labels)
        labels = sorted(set(all_true_labels) | set(all_pred_labels))
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.title('Confusion Matrix - Image Based Evaluation')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.show()
    
    # 結果の詳細表示
    results_df = pd.DataFrame(detailed_results)
    print("\n詳細結果:")
    print(results_df.to_string(index=False))
    
    return all_true_labels, all_pred_labels, results_df

# 実行
true_labels, pred_labels, results_df = evaluate_image_recognition(db_base_path="./")


face_recognitionによる予測

face rekognitionのライブラリを当初は使う予定だった。正しいかのような問題が起こった<br>
・pip installするときにdlibライブラリを合わせてインストールする必要があり、C++の環境が必要でインストールが大変<br>
・pip installできてもライブラリをimportする際に、別ライブラリを入れる必要があるエラーが起こった<br>
利用までがかなり煩雑であるため、Deepfaceを使う方針へ転換

In [ ]:
import face_recognition
import cv2

# Open the input movie file
input_movie = cv2.VideoCapture("hamilton_clip.mp4")
length = int(input_movie.get(cv2.CAP_PROP_FRAME_COUNT))

# Create an output movie file (make sure resolution/frame rate matches input video!)
fourcc = cv2.VideoWriter_fourcc(*'XVID')
output_movie = cv2.VideoWriter('output.avi', fourcc, 29.97, (640, 360))

# Load some sample pictures and learn how to recognize them.
lmm_image = face_recognition.load_image_file("lin-manuel-miranda.png")
lmm_face_encoding = face_recognition.face_encodings(lmm_image)[0]

al_image = face_recognition.load_image_file("alex-lacamoire.png")
al_face_encoding = face_recognition.face_encodings(al_image)[0]

known_faces = [
    lmm_face_encoding,
    al_face_encoding
]

# Initialize some variables
face_locations = []
face_encodings = []
face_names = []
frame_number = 0

while True:
    # Grab a single frame of video
    ret, frame = input_movie.read()
    frame_number += 1

    # Quit when the input video file ends
    if not ret:
        break

    # Convert the image from BGR color (which OpenCV uses) to RGB color (which face_recognition uses)
    rgb_frame = frame[:, :, ::-1]

    # Find all the faces and face encodings in the current frame of video
    face_locations = face_recognition.face_locations(rgb_frame)
    face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)

    face_names = []
    for face_encoding in face_encodings:
        # See if the face is a match for the known face(s)
        match = face_recognition.compare_faces(known_faces, face_encoding, tolerance=0.50)

        # If you had more than 2 faces, you could make this logic a lot prettier
        # but I kept it simple for the demo
        name = None
        if match[0]:
            name = "Lin-Manuel Miranda"
        elif match[1]:
            name = "Alex Lacamoire"

        face_names.append(name)

    # Label the results
    for (top, right, bottom, left), name in zip(face_locations, face_names):
        if not name:
            continue

        # Draw a box around the face
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)

        # Draw a label with a name below the face
        cv2.rectangle(frame, (left, bottom - 25), (right, bottom), (0, 0, 255), cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        cv2.putText(frame, name, (left + 6, bottom - 6), font, 0.5, (255, 255, 255), 1)

    # Write the resulting image to the output video file
    print("Writing frame {} / {}".format(frame_number, length))
    output_movie.write(frame)

# All done!
input_movie.release()
cv2.destroyAllWindows()

Googleだと以下のエラーが出た<br>
TypeError: 'NoneType' object is not iterable<br>
おそらく、スクレイピングをしようと思った際にパース（解析）処理の例外エラーが発生したものと思われる。そのため今回はBingを使うのが良いと判断

In [9]:
crawler = GoogleImageCrawler(storage={'root_dir' : './data_google'})
crawler.crawl(keyword = keyword, max_num = photo_num)

2026-02-24 22:58:49,562 - INFO - icrawler.crawler - start crawling...
2026-02-24 22:58:49,564 - INFO - icrawler.crawler - starting 1 feeder threads...
2026-02-24 22:58:49,567 - INFO - feeder - thread feeder-001 exit
2026-02-24 22:58:49,569 - INFO - icrawler.crawler - starting 1 parser threads...
2026-02-24 22:58:49,573 - INFO - icrawler.crawler - starting 1 downloader threads...
2026-02-24 22:58:50,250 - INFO - parser - parsing result page https://www.google.com/search?q=%E8%B3%80%E5%96%9C%E9%81%A5%E9%A6%99&ijn=0&start=0&tbs=&tbm=isch
Exception in thread parser-001:
Traceback (most recent call last):
  File "C:\Users\真希\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\真希\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\真希\Downloads\nogi_movie_naming\venv\Lib\site-pa